# Create Conrad N. Hilton Foundation Grant Awards (GRANT PATTERN, Method 5 WP REST + HTML)

The Conrad N. Hilton Foundation is a large US private foundation funding humanitarian work across program areas including Safe Water, Homelessness, Foster Youth, Catholic Sisters, Early Childhood Development, Avoidable Blindness, Multiple Sclerosis, Opportunity Youth, Disaster Relief & Recovery, and the Hilton Humanitarian Prize. This ingest covers the foundation's **public grants archive** (`hiltonfoundation.org/grant/...`), one row per grant.

**Method 5 (WP REST + static HTML).** The scraper (`scripts/local/hilton_to_s3.py`) enumerates every grant from the WordPress `grant` custom-post-type REST endpoint (`/wp-json/wp/v2/grant?per_page=100&page=N`), which gives the recipient organization (post title), the slug, the landing URL, and taxonomy terms (`program-area`, `area`, `grant_year`). The dollar amount and project dates are NOT in the REST payload (`content.rendered` is empty; ACF fields aren't surfaced) — they are parsed from each grant detail page's `<ul class="grant-info-list">` sidebar (`<li><strong>Label:</strong> value</li>` rows). Plain `requests` + BeautifulSoup; no browser automation.

**Awarding body:** Conrad N. Hilton Foundation - F4320306180 (US, ROR 05g9snv96, DOI 10.13039/100000910)

**Source:** `hiltonfoundation.org/grant/{slug}`. This is an **org-level grant funder** — each grant is led by the grantee organization (no named principal investigator). Each grant carries a **Grantee Name** (recipient org), an optional **Project Description**, a **Program** (program-area), an **Area Served** (region), and on most pages a **Grant Amount** (USD), **Awarded Date**, **Project Start Date**, **Project End Date**, and **Term (Months)**.

**Schema choices:**
- One row per grant. `funder_award_id` = the source URL slug, e.g. `world-resources-institute-2` (stable, unique, source-authoritative).
- `funding_type` = `grant` uniformly.
- `funder_scheme` = the grant's **Program** (program-area: Safe Water, Homelessness, Foster Youth, Catholic Sisters, Avoidable Blindness, Multiple Sclerosis, Opportunity Youth, Disaster Relief & Recovery, Hilton Humanitarian Prize, ...).
- **Amount IS published on ~100% of grants in this archive** (`$3000000` form) → `amount` (DOUBLE) + `currency` = `USD` where present, NULL otherwise. **§6.7 is NOT waived** — the figure is populated wherever the source posts one, never imputed. (Any `$0` is treated as NULL, not a real award amount.)
- **Coverage caveat:** the foundation's public archive exposes only recent grants (start years 2021-2026); its full historical record is much larger but not reachable from the public site (same recency limitation as Doris Duke #143).
- `lead_investigator` = an **org-only** lead: `given_name`/`family_name` are NULL and `affiliation.name` = the grantee organization. `affiliation.country` is NULL — the source's "Area Served" is the region the work serves (e.g. Africa), NOT the grantee's country, so country is never guessed (CLAUDE.md).
- `co_lead_investigator` and `investigators` are NULL (the source names no individuals).
- `start_year` parsed from the Project Start Date (falling back to Awarded Date); `end_year` from the Project End Date; `start_date` / `end_date` left NULL (the source gives month precision, not day, so no false-precision dates are asserted).
- `description` = the grant's Project Description.

**S3 location:** `s3a://openalex-ingest/awards/hilton/hilton_grants.parquet`

**Provenance:** `hilton_foundation` (verified count=0 on production 2026-05-29).

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hilton_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hilton/hilton_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.hilton_raw;

## Step 1.5: Inspect raw + program/year/coverage

Expected: 2,551 grants (the foundation's public archive exposes only recent grants, 2021-2026). title (grantee_org) / slug / amount / start_year / description 100%; program 91.5%; end_year 99.9%.

In [ ]:
%sql
DESCRIBE openalex.awards.hilton_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hilton_raw LIMIT 5;

In [ ]:
%sql
-- Per-(program, start_year) row counts + coverage.
SELECT program, start_year, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       COUNT(grantee_org) AS has_org
FROM openalex.awards.hilton_raw
GROUP BY program, start_year
ORDER BY start_year DESC, rows DESC;

In [ ]:
%sql
-- Top grantee organizations (sanity check parsing didn't bleed fields).
SELECT grantee_org, COUNT(*) AS rows
FROM openalex.awards.hilton_raw
WHERE grantee_org IS NOT NULL
GROUP BY grantee_org
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify Conrad N. Hilton Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306180;

## Step 2: Transform to award schema

Per-row `display_name` = `{program} - {grantee_org} ({start_year})`. `lead_investigator` is an org-only lead: `given_name`/`family_name` NULL, `affiliation.name` = grantee org, `affiliation.country` NULL (Area Served is a region the work serves, not the grantee's country). `co_lead_investigator` / `investigators` are NULL.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hilton_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306180  -- Conrad N. Hilton Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(COALESCE(n.program, 'Grant'), ' - ',
           COALESCE(n.grantee_org, n.title),
           CASE WHEN n.start_year IS NOT NULL THEN CONCAT(' (', n.start_year, ')') ELSE '' END) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN n.currency ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.program AS funder_scheme,
    'hilton_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    TRY_CAST(n.end_year AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.grantee_org AS name,
                CAST(NULL AS STRING) AS country,  -- Area Served is a region, not the grantee's country; never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.hilton_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 153

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hilton_foundation' AND priority = 153;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    153 AS priority  -- Conrad N. Hilton Foundation grants
FROM openalex.awards.hilton_awards;

## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived): every grant in this 2021-2026 archive publishes a dollar figure, so `pct_amount` ≈ 100% (the `$0`-guard nulls any placeholder, so it may sit marginally under 100%). Other completeness checks: grantee_org / display_name / description / start_year ~100%, program ~91.5%, currency = [USD].

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.hilton_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.hilton_awards;

In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.hilton_awards;

In [ ]:
%sql
-- §6.7 amount check (NOT waived, partial). Expect currency = [USD].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.hilton_awards;

In [ ]:
%sql
-- Program (scheme) split.
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(amount) AS has_amount,
       ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.hilton_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.hilton_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, start_year, end_year, amount, currency,
       lead_investigator.affiliation.name AS org
FROM openalex.awards.hilton_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;